# Eksperimen SML — Telco Churn
**Nama:** Epafraditus Memoriano  
**Kriteria 1:** Eksperimen manual — data loading, EDA, dan preprocessing.

Dataset: `telco_churn_raw.csv` (klasifikasi biner: pelanggan churn / tidak).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 1. Data Loading

In [ ]:
df = pd.read_csv('telco_churn_raw.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# TotalCharges tersimpan sebagai object karena ada nilai kosong -> paksa numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.describe()

In [ ]:
# Missing values
missing = df.isna().sum()
print(missing[missing > 0])

In [ ]:
# Distribusi target
ax = df['Churn'].value_counts().plot(kind='bar', color=['#4c72b0', '#dd8452'])
ax.set_title('Distribusi Churn (0 = tetap, 1 = churn)')
ax.set_xlabel('Churn'); ax.set_ylabel('Jumlah')
plt.tight_layout(); plt.show()
print(df['Churn'].value_counts(normalize=True).round(3))

In [ ]:
# Distribusi fitur numerik
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'NumServices']
df[num_cols].hist(figsize=(10, 6), bins=30)
plt.tight_layout(); plt.show()

In [ ]:
# Korelasi fitur numerik terhadap target
corr = df[num_cols + ['TechSupport', 'Churn']].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation matrix'); plt.tight_layout(); plt.show()

In [ ]:
# Churn rate per tipe kontrak (kategori penting)
rate = df.groupby('Contract')['Churn'].mean().sort_values()
rate.plot(kind='barh', color='#55a868')
plt.title('Churn rate per Contract'); plt.xlabel('Churn rate')
plt.tight_layout(); plt.show()
rate

**Temuan EDA:**
- Target sedikit tidak seimbang → gunakan `stratify` saat split.
- `TotalCharges` punya sebagian nilai kosong → imputasi median.
- Kontrak *Month-to-month* punya churn rate jauh lebih tinggi → fitur kategorikal penting, perlu encoding.
- Skala fitur numerik berbeda-beda → perlu standardisasi.

## 3. Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

TARGET = 'Churn'
NUMERIC = ['tenure', 'MonthlyCharges', 'TotalCharges', 'TechSupport', 'NumServices']
CATEGORICAL = ['Contract', 'PaymentMethod']

X = df.drop(columns=['customerID', TARGET])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, ' Test:', X_test.shape)

In [ ]:
numeric = Pipeline([('impute', SimpleImputer(strategy='median')),
                    ('scale', StandardScaler())])
categorical = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
ct = ColumnTransformer([('num', numeric, NUMERIC),
                        ('cat', categorical, CATEGORICAL)])

X_train_t = ct.fit_transform(X_train)
X_test_t = ct.transform(X_test)
cols = list(ct.get_feature_names_out())
X_train_df = pd.DataFrame(X_train_t, columns=cols)
X_test_df = pd.DataFrame(X_test_t, columns=cols)
X_train_df.head()

In [ ]:
# Simpan dataset hasil preprocessing
import os
os.makedirs('telco_churn_preprocessing', exist_ok=True)
train_out = X_train_df.copy(); train_out[TARGET] = y_train.reset_index(drop=True)
test_out = X_test_df.copy(); test_out[TARGET] = y_test.reset_index(drop=True)
train_out.to_csv('telco_churn_preprocessing/train.csv', index=False)
test_out.to_csv('telco_churn_preprocessing/test.csv', index=False)
print('Saved. Train cols:', list(train_out.columns))

## 4. Otomatisasi
Seluruh langkah preprocessing di atas dikemas ulang menjadi fungsi `preprocess()` di `automate_Epafraditus-Memoriano.py` sehingga bisa dipanggil ulang / dijalankan otomatis oleh GitHub Actions.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('automate', 'automate_Epafraditus-Memoriano.py')
automate = importlib.util.module_from_spec(spec); spec.loader.exec_module(automate)
Xtr, Xte, ytr, yte = automate.preprocess('telco_churn_raw.csv')
print('automate.preprocess OK ->', Xtr.shape, Xte.shape)